In [2]:
# Run this in your notebook to verify what the model SHOULD predict
import pickle, numpy as np

with open('../models/flood_model_stacking_ensemble.pkl', 'rb') as f:
    saved = pickle.load(f)

model = saved['model']
le    = saved['label_encoder']

# Test 1: February, dry season, Darbhanga Bihar (should be LOW)
# Test 2: July monsoon, Kerala (should be HIGH)
test_cases = [
    # lat,  lon,   rain, temp, hum,  discharge, wlevel, soil,  pressure, et0
    [26.15, 85.89, 0.0,  18.0, 65.0, 50.0,     100.0,  14.0,  1013.0,   1.5],  # Feb dry
    [10.00, 76.32, 180.0,28.0, 92.0, 500.0,    800.0,  30.0,  1002.0,   0.8],  # Jul monsoon
]

for i, t in enumerate(test_cases):
    import numpy as np
    log_rain = np.log1p(t[2])
    log_dis  = np.log1p(t[4+1])
    log_wl   = np.log1p(t[4+2])
    # just raw - no scaling
    X = np.array([[t[0],t[1],log_rain,t[3],t[4],log_dis,log_wl,t[7],t[8],t[9],0,0,0,0,0,0,0,0] + [0]*117], dtype=np.float32)
    pred = le.inverse_transform(model.predict(X))
    proba = model.predict_proba(X)[0]
    print(f"Test {i+1}: {pred[0]} | probs: {dict(zip(le.classes_, proba.round(3)))}")

Test 1: High | probs: {'High': 1.0, 'Low': 0.0, 'Moderate': 0.0}
Test 2: High | probs: {'High': 1.0, 'Low': 0.0, 'Moderate': 0.0}


In [3]:
# Paste this in your notebook AFTER all cells have run
# This tests with PROPERLY scaled + encoded features

# Test with actual training samples — grab 3 rows from each class
for cls in ['Low', 'Moderate', 'High']:
    idx = np.where(le.transform(df_filled['flood_risk'].values) == le.transform([cls])[0])[0][:3]
    preds = le.inverse_transform(final_model.predict(X[idx]))
    print(f"True: {cls} → Predicted: {list(preds)}")

# Also check raw probabilities on a known Low sample
low_idx = np.where(le.transform(df_filled['flood_risk'].values) == le.transform(['Low'])[0])[0][0]
print(f"\nLow sample proba: {dict(zip(le.classes_, final_model.predict_proba(X[low_idx:low_idx+1])[0].round(3)))}")

NameError: name 'df_filled' is not defined